# Chapter 16: String and Math Algorithms

Find a pattern inside text, reuse partial matches, update rolling fingerprints, and perform efficient integer calculations. All seven exercises from the supplied chapter have runnable worked solutions.

Select **JavaScript (Node.js)**. Each code cell is self-contained: it includes its own definitions and assertions, and can be run independently or rerun. Run all cells to see the full lesson. No packages are required. Assertions throw if a result differs from its expected value; otherwise they stay silent.

For a single-pattern search, return the first zero-based index, −1 if absent, and 0 for an empty pattern. Let n be text length and m pattern length. JavaScript indices count UTF-16 code units, not visible characters; traces use ASCII. These are exact, case-sensitive searches without Unicode normalization.

## 1. Naïve substring search

Try each start from 0 through n − m. Compare the pattern from left to right; on a mismatch, move the start one position to the right. A full match returns immediately. An empty pattern matches at index 0 without comparing characters.

For 1 ≤ m ≤ n, at most m(n − m + 1) character comparisons are made: one complete pattern per possible start. The ordinary index-only algorithm uses O(1) extra space. This instrumented version stores O(n) alignment records so you can inspect the work.

### Exercise 1: Count comparisons
Search for AAAB, then BAAA, in AAAAAA. Predict the tried starts and comparison totals.

**Read the output:** AAAB matches three As before failing on B at each of starts 0, 1, and 2: 12 comparisons. BAAA fails immediately at each start: 3 comparisons. Both return −1. The helper `naiveSearch` returns only the index, while `naiveTrace` also returns the recorded counts.

In [ ]:
(() => {
const assert = require('node:assert/strict');
function naiveTrace(text, pattern) {
  let comparisons = 0;
  const alignments = [];
  for (let start = 0; start <= text.length - pattern.length; start++) {
    let j = 0, count = 0;
    while (j < pattern.length) {
      comparisons++; count++;
      if (text[start + j] !== pattern[j]) break;
      j++;
    }
    alignments.push({ start, comparisons: count });
    if (j === pattern.length) return { index: start, comparisons, alignments };
  }
  return { index: -1, comparisons, alignments };
}
function naiveSearch(text, pattern) { return naiveTrace(text, pattern).index; }
for (const [pattern, total] of [['AAAB', 12], ['BAAA', 3]]) {
  const result = naiveTrace('AAAAAA', pattern);
  console.log('Pattern:', pattern, 'index:', result.index, 'comparisons:', result.comparisons);
  console.table(result.alignments);
  assert.deepEqual(result.alignments.map(row => row.start), [0, 1, 2]);
  assert.equal(result.comparisons, total);
  assert.equal(result.index, -1);
}
assert.equal(naiveSearch('BANANA', 'ANA'), 1);
})();

## 2. KMP: reuse a matched prefix

A prefix table π stores, at each position, the length of the longest proper prefix that is also a suffix of the segment ending there. “Proper” excludes the entire segment. For ABABA the entries are [0, 0, 1, 2, 3].

During search, j is the number of matched pattern characters. A mismatch after ABAB does not erase everything: its ending AB is also the pattern's beginning, so fall back to π[3] = 2. Compare the same text character again at the shorter matched length. The text position never moves backward.

### Exercise 2: Build a prefix table
Build π for ABABA and identify the fallback after four matches. The cell also traces ABABC inside ABABABC. Watch text index 4 appear twice: the mismatch against C falls back to j = 2, where A matches. The result starts at index 2.

Preprocessing takes O(m), scanning O(n), and the table uses O(m) extra space. Matched length rises at most once per text character; every fallback lowers it, bounding total fallback work. The optional trace adds O(n) records. Its single-comparison loop makes each recorded row one actual character comparison; the compact chapter search may evaluate equality more than once.

In [ ]:
(() => {
const assert = require('node:assert/strict');
function prefixTable(pattern) {
  const pi = Array(pattern.length).fill(0);
  for (let i = 1, j = 0; i < pattern.length; i++) {
    while (j > 0 && pattern[i] !== pattern[j]) j = pi[j - 1];
    if (pattern[i] === pattern[j]) j++;
    pi[i] = j;
  }
  return pi;
}
function kmpSearch(text, pattern) {
  if (pattern.length === 0) return 0;
  const pi = prefixTable(pattern);
  for (let i = 0, j = 0; i < text.length; i++) {
    while (j > 0 && text[i] !== pattern[j]) j = pi[j - 1];
    if (text[i] === pattern[j]) j++;
    if (j === pattern.length) return i - j + 1;
  }
  return -1;
}
function kmpTrace(text, pattern) {
  if (!pattern.length) return { index: 0, events: [] };
  const pi = prefixTable(pattern), events = [];
  let i = 0, j = 0;
  while (i < text.length) {
    const matches = text[i] === pattern[j];
    const nextMatched = matches ? j + 1 : j > 0 ? pi[j - 1] : 0;
    events.push({ textIndex: i, character: text[i], patternIndex: j,
      matches, nextMatched });
    if (matches) {
      i++; j++;
      if (j === pattern.length) return { index: i - j, events };
    } else if (j > 0) j = pi[j - 1];
    else i++;
  }
  return { index: -1, events };
}
assert.deepEqual(prefixTable('ABABA'), [0, 0, 1, 2, 3]);
assert.equal(prefixTable('ABABA')[3], 2);
console.table([... 'ABABA'].map((character, index) => ({ index, character, pi: prefixTable('ABABA')[index] })));
const result = kmpTrace('ABABABC', 'ABABC');
console.table(result.events);
assert.equal(result.index, 2);
assert.equal(kmpSearch('ABABABC', 'ABABC'), 2);
assert.ok(result.events.some(row => row.textIndex === 4 && !row.matches && row.nextMatched === 2));
})();

## 3. Boyer–Moore's bad-character rule

Compare each aligned window from right to left. On a mismatch at pattern index j with text character c, shift by max(1, j − last(c)); last(c) is its rightmost pattern position or −1 when absent. The minimum shift of 1 ensures progress even if that character appears to the right of j.

For NEEDLE in XXXXXXNEEDLE, the rightmost E first meets X. X is absent, so shift 5 − (−1) = 6, landing directly on the match. This can skip many starts without examining them.

### Exercise 3: Find the shift
For ABCA and a mismatch at j = 3, X gives shift 4; B, last seen at index 1, gives shift 2.

This is the bad-character variant, with O(m) preprocessing, O(nm) worst-case search, and one Map entry per distinct pattern character, assuming constant-time Map operations. Full Boyer–Moore also uses the **good-suffix rule**: reuse the suffix already matched by aligning another compatible occurrence, or a prefix matching its end. It selects the larger safe shift. The implementation below implements only the bad-character rule.

In [ ]:
(() => {
const assert = require('node:assert/strict');
function boyerMooreBadCharacter(text, pattern) {
  const m = pattern.length;
  if (m === 0) return 0;
  const last = new Map();
  for (let j = 0; j < m; j++) last.set(pattern[j], j);
  let start = 0;
  while (start <= text.length - m) {
    let j = m - 1;
    while (j >= 0 && pattern[j] === text[start + j]) j--;
    if (j < 0) return start;
    start += Math.max(1, j - (last.get(text[start + j]) ?? -1));
  }
  return -1;
}
const pattern = 'ABCA';
const last = new Map([...pattern].map((character, index) => [character, index]));
const shifts = ['X', 'B'].map(character => ({ character,
  last: last.get(character) ?? -1, shift: Math.max(1, 3 - (last.get(character) ?? -1)) }));
console.table(shifts);
assert.deepEqual(shifts.map(row => row.shift), [4, 2]);
assert.equal(boyerMooreBadCharacter('XXXXXXNEEDLE', 'NEEDLE'), 6);
console.log('NEEDLE starts at:', boyerMooreBadCharacter('XXXXXXNEEDLE', 'NEEDLE'));
})();

## 4. Rabin–Karp: roll a fingerprint

A polynomial hash reads character codes like digits in base b, reducing modulo q after each update. For a window of length m, factor = bᵐ⁻¹ mod q is the weight of its first character. Subtract the outgoing contribution, multiply by b to shift the remaining digits, and add the incoming character.

JavaScript `%` can give a negative remainder; `mod` normalizes it into [0, q). The production example uses UTF-16 codes, base 257, and modulus 1,000,003. These constants keep intermediate integer arithmetic within JavaScript's exact safe-integer range. Changing them requires checking that assumption.

### Exercise 4: Spot a collision
Using A = 1, B = 2, base 10 and modulus 3, AB and BA both hash to 0. Equal fingerprints are candidates, not proof of equality: character verification rejects BA when the pattern is AB. The toy examples also roll AB to BC modulo 7: 5 becomes 2.

The index-only search uses O(1) extra space. If c windows trigger verification, time is O(n + m + cm), with O(nm) worst case for nonempty patterns no longer than the text. Each roll itself is O(1); `startsWith` verifies actual characters before accepting.

In [ ]:
(() => {
const assert = require('node:assert/strict');
function rabinKarpSearch(text, pattern) {
  const n = text.length, m = pattern.length;
  if (m === 0) return 0;
  if (m > n) return -1;
  const base = 257, modulus = 1_000_003;
  const mod = x => ((x % modulus) + modulus) % modulus;
  let factor = 1, target = 0, window = 0;
  for (let j = 0; j < m; j++) {
    target = mod(target * base + pattern.charCodeAt(j));
    window = mod(window * base + text.charCodeAt(j));
    if (j < m - 1) factor = mod(factor * base);
  }
  for (let i = 0; i <= n - m; i++) {
    if (window === target && text.startsWith(pattern, i)) return i;
    if (i < n - m) {
      window = mod((window - text.charCodeAt(i) * factor) * base
        + text.charCodeAt(i + m));
    }
  }
  return -1;
}
function toyHash(word, modulus) {
  let hash = 0;
  for (const character of word) hash = (hash * 10 + character.charCodeAt(0) - 64) % modulus;
  return hash;
}
assert.equal(toyHash('AB', 3), 0);
assert.equal(toyHash('BA', 3), 0);
assert.equal('BA'.startsWith('AB'), false);
console.log('Collision:', { AB: toyHash('AB', 3), BA: toyHash('BA', 3), verified: 'BA'.startsWith('AB') });
const mod7 = value => ((value % 7) + 7) % 7;
const factor = 10 % 7;
const rolled = mod7((toyHash('AB', 7) - 1 * factor) * 10 + 3);
assert.equal(rolled, toyHash('BC', 7));
assert.equal(rolled, 2);
assert.equal(rabinKarpSearch('BANANA', 'ANA'), 1);
console.log('Rolling AB to BC:', { old: toyHash('AB', 7), factor, rolled });
})();

## 5. Automata and dictionary matching

A deterministic finite automaton has one next state for each current-state/input-symbol pair. For pattern search a state can represent how many characters match a suffix of the text seen so far. KMP calculates fallbacks as needed; a dense DFA precomputes all state/alphabet transitions using O(m|Σ|) space.

For many patterns, **Aho–Corasick** builds a trie sharing common prefixes, then adds failure links to shorter suffix states. A state can report its own terminal pattern and patterns ending through its failure chain. This finds overlapping dictionary matches in a single pass.

### Exercise 5: Trace dictionary matching
For HE, SHE, and HERS in USHERS, report each match with inclusive start and end indices. SHE spans 1–3, HE spans 2–3, and HERS spans 2–5. SHE and HE share end index 3.

The cell uses sparse Maps and copies inherited outputs for readability. A breadth-first queue establishes failure links only after shallower links are ready; a head index avoids costly queue shifts. Repeated dictionary entries are deduplicated, and empty patterns are rejected to keep output semantics simple. With expected O(1) Map access, scanning is O(n + z), where z is matches reported, and the returned results use O(z) space. Output copying can make construction storage superlinear in total pattern length L. A dense-transition implementation with output links instead has O(L|Σ|) construction time and space; those are not the construction bounds of this teaching implementation.

In [ ]:
(() => {
const assert = require('node:assert/strict');
function buildDictionary(patterns) {
  if (patterns.some(pattern => pattern.length === 0)) {
    throw new RangeError('Use nonempty dictionary patterns.');
  }
  const nodes = [{ next: new Map(), fail: 0, output: [] }];
  for (const pattern of new Set(patterns)) {
    let state = 0;
    for (let i = 0; i < pattern.length; i++) {
      const symbol = pattern[i];
      if (!nodes[state].next.has(symbol)) {
        nodes[state].next.set(symbol, nodes.length);
        nodes.push({ next: new Map(), fail: 0, output: [] });
      }
      state = nodes[state].next.get(symbol);
    }
    nodes[state].output.push(pattern);
  }
  const queue = [...nodes[0].next.values()];
  for (let head = 0; head < queue.length; head++) {
    const state = queue[head];
    for (const [symbol, child] of nodes[state].next) {
      queue.push(child);
      let fallback = nodes[state].fail;
      while (fallback && !nodes[fallback].next.has(symbol)) fallback = nodes[fallback].fail;
      nodes[child].fail = nodes[fallback].next.get(symbol) ?? 0;
      nodes[child].output.push(...nodes[nodes[child].fail].output);
    }
  }
  return nodes;
}
function dictionarySearch(text, nodes) {
  const matches = [];
  let state = 0;
  for (let i = 0; i < text.length; i++) {
    const symbol = text[i];
    while (state && !nodes[state].next.has(symbol)) state = nodes[state].fail;
    state = nodes[state].next.get(symbol) ?? 0;
    for (const pattern of nodes[state].output) {
      matches.push({ pattern, start: i - pattern.length + 1, end: i });
    }
  }
  return matches;
}
const nodes = buildDictionary(['HE', 'SHE', 'HERS']);
const matches = dictionarySearch('USHERS', nodes);
console.table(matches);
assert.deepEqual(matches, [
  { pattern: 'SHE', start: 1, end: 3 },
  { pattern: 'HE', start: 2, end: 3 },
  { pattern: 'HERS', start: 2, end: 5 }
]);
assert.equal(matches[0].end, matches[1].end);
assert.equal(dictionarySearch('ZZZ', nodes).length, 0);
assert.deepEqual(dictionarySearch('AAA', buildDictionary(['A', 'AA'])).map(x => x.pattern),
  ['A', 'AA', 'A', 'AA', 'A']);
assert.throws(() => buildDictionary(['']), RangeError);
})();

## 6. Preprocess the text: suffix indexes

If the text stays fixed while queries change, index its suffixes. A **suffix tree** is a compressed trie of all suffixes (usually with a unique terminator): edges represent text spans instead of single characters. With constant-time edge lookup, following a query costs O(m), plus O(z) to report matches. Efficient construction is a separate algorithm.

A **suffix array** stores suffix starting positions in lexicographic order. Suffixes beginning with a given pattern occupy one contiguous interval. Two binary searches find its lower and upper boundaries in O(m log n) character-comparison time; reporting costs O(z).

### Exercise 6: Search BANANA's suffix array
Sorted suffix positions are [5, 3, 1, 0, 4, 2], representing A, ANA, ANANA, BANANA, NA, NANA. ANA matches starts [3, 1] in suffix-array order; sort those indices numerically if text order is needed.

The builder below deliberately materializes suffix strings and uses comparison sorting: conservatively O(n²) character storage and O(n² log n) construction time under a comparison-sort model. It demonstrates the index rather than an advanced construction algorithm. Queries compare characters directly, without allocating suffix strings. The index omits the empty suffix; therefore an empty-pattern query reports its n stored positions, while the single-pattern functions return the first empty match, index 0.

In [ ]:
(() => {
const assert = require('node:assert/strict');
function suffixArray(text) {
  // Straightforward teaching construction, not a linear-time suffix-array builder.
  return Array.from({ length: text.length }, (_, index) => ({ index, suffix: text.slice(index) }))
    .sort((a, b) => a.suffix < b.suffix ? -1 : a.suffix > b.suffix ? 1 : 0)
    .map(item => item.index);
}
function suffixMatches(text, indices, pattern) {
  function compare(start) {
    for (let j = 0; j < pattern.length; j++) {
      if (start + j === text.length) return -1;
      if (text[start + j] < pattern[j]) return -1;
      if (text[start + j] > pattern[j]) return 1;
    }
    return 0;
  }
  function bound(upper) {
    let low = 0, high = indices.length;
    while (low < high) {
      const mid = low + Math.floor((high - low) / 2);
      const order = compare(indices[mid]);
      if (order < 0 || (upper && order === 0)) low = mid + 1;
      else high = mid;
    }
    return low;
  }
  return indices.slice(bound(false), bound(true));
}
const text = 'BANANA';
const indices = suffixArray(text);
console.table(indices.map(index => ({ index, suffix: text.slice(index) })));
assert.deepEqual(indices, [5, 3, 1, 0, 4, 2]);
const matches = suffixMatches(text, indices, 'ANA');
assert.deepEqual(matches, [3, 1]);
assert.deepEqual(suffixMatches(text, indices, 'APPLE'), []);
console.log('ANA starts, in suffix-array order:', matches);
})();

## 7. Euclid's algorithm and modular exponentiation

**GCD:** Replace (a, b) with (b, a mod b) until b becomes zero. Common divisors are preserved because a = qb + r. The remaining a is the greatest common divisor. For positive inputs there are O(log min(a, b) + 1) remainder steps; BigInt operation costs also grow with operand bit length. We require nonnegative BigInts and use the programming convention gcd(0, 0) = 0.

**Modular exponentiation:** Compute baseᵉ mod q without first creating the enormous power. An odd exponent contributes one copy of the current base; then square the base modulo q and halve the exponent using integer division. The invariant is result × baseᵉ congruent to the original requested power modulo q. Residues stay below q after reduction, although multiplication intermediates can approach q².

For e ≥ 1, there are floor(log₂ e) + 1 iterations, each with one square and at most one extra multiplication. This counts arithmetic operations, not constant-time machine instructions: BigInt multiplication and division depend on digit count. Inputs require exponent ≥ 0 and modulus > 0. A zero exponent returns 1 mod q, including 0 when q = 1.

### Exercise 7: Trace remainder and squaring steps
For gcd(84n, 30n), the remainders are 24, 6, 0 and the result is 6. For modPow(3n, 13n, 7n), record exponent, base, and result **before** each iteration: (13, 3, 1), (6, 2, 3), (3, 4, 3), (1, 2, 5). The final residue is 3. JavaScript's `n` suffix denotes BigInt; do not mix Number and BigInt arithmetic.

In [ ]:
(() => {
const assert = require('node:assert/strict');
function gcd(a, b) {
  if (typeof a !== 'bigint' || typeof b !== 'bigint') throw new TypeError('Use BigInt inputs.');
  if (a < 0n || b < 0n) throw new RangeError('Use nonnegative inputs.');
  while (b !== 0n) [a, b] = [b, a % b];
  return a;
}
function modPow(base, exponent, modulus) {
  if ([base, exponent, modulus].some(value => typeof value !== 'bigint')) {
    throw new TypeError('Use BigInt inputs.');
  }
  if (exponent < 0n || modulus <= 0n) throw new RangeError('Require exponent >= 0 and modulus > 0.');
  base = ((base % modulus) + modulus) % modulus;
  let result = 1n % modulus;
  while (exponent > 0n) {
    if (exponent % 2n === 1n) result = result * base % modulus;
    base = base * base % modulus;
    exponent /= 2n;
  }
  return result;
}
let a = 84n, b = 30n;
const remainders = [];
while (b !== 0n) {
  const remainder = a % b;
  remainders.push({ a: String(a), b: String(b), remainder: String(remainder) });
  [a, b] = [b, remainder];
}
console.table(remainders);
assert.deepEqual(remainders.map(row => row.remainder), ['24', '6', '0']);
assert.equal(gcd(84n, 30n), 6n);
let base = 3n, exponent = 13n, result = 1n;
const states = [];
while (exponent > 0n) {
  states.push([exponent, base, result]);
  if (exponent % 2n === 1n) result = result * base % 7n;
  base = base * base % 7n;
  exponent /= 2n;
}
console.table(states.map(([exponent, base, result]) => ({
  exponent: String(exponent), base: String(base), result: String(result) })));
assert.deepEqual(states, [[13n, 3n, 1n], [6n, 2n, 3n], [3n, 4n, 3n], [1n, 2n, 5n]]);
assert.equal(result, 3n);
assert.equal(modPow(3n, 13n, 7n), 3n);
console.log('GCD:', String(a), 'Final modular power:', String(result));
})();

## 8. Choose what to preprocess

| Workload | Approach | What is prepared |
| --- | --- | --- |
| Small text or short pattern | Naïve search | Nothing |
| One pattern with worst-case linear search | KMP | Pattern prefix table |
| One pattern with opportunities for large skips | Bad-character Boyer–Moore | Last occurrence of each character |
| Rolling window fingerprints | Rabin–Karp | Initial hashes and outgoing-character factor |
| Many patterns in one text pass | Aho–Corasick | Dictionary trie and failure links |
| Many queries over fixed text | Suffix array / tree | Text suffix index |

The GCD and modular-power examples connect string hashing to modular arithmetic. Rabin–Karp needs bᵐ⁻¹ mod q; its initial hash loop computes that factor incrementally, though modular exponentiation can compute it independently.

## 9. Verification

The final cell is independent of every earlier cell. It checks the four single-pattern algorithms against JavaScript `indexOf`, including empty inputs, overlaps, missing patterns, and UTF-16 indices. It also checks prefix-table invariants, rolling updates, dictionary results against brute-force matches, suffix queries, and mathematical edge cases. Seeded random cases make failures reproducible.

In [ ]:
(() => {
const assert = require('node:assert/strict');
function naiveTrace(text, pattern) {
  let comparisons = 0;
  const alignments = [];
  for (let start = 0; start <= text.length - pattern.length; start++) {
    let j = 0, count = 0;
    while (j < pattern.length) {
      comparisons++; count++;
      if (text[start + j] !== pattern[j]) break;
      j++;
    }
    alignments.push({ start, comparisons: count });
    if (j === pattern.length) return { index: start, comparisons, alignments };
  }
  return { index: -1, comparisons, alignments };
}
function naiveSearch(text, pattern) { return naiveTrace(text, pattern).index; }

function prefixTable(pattern) {
  const pi = Array(pattern.length).fill(0);
  for (let i = 1, j = 0; i < pattern.length; i++) {
    while (j > 0 && pattern[i] !== pattern[j]) j = pi[j - 1];
    if (pattern[i] === pattern[j]) j++;
    pi[i] = j;
  }
  return pi;
}
function kmpSearch(text, pattern) {
  if (pattern.length === 0) return 0;
  const pi = prefixTable(pattern);
  for (let i = 0, j = 0; i < text.length; i++) {
    while (j > 0 && text[i] !== pattern[j]) j = pi[j - 1];
    if (text[i] === pattern[j]) j++;
    if (j === pattern.length) return i - j + 1;
  }
  return -1;
}
function kmpTrace(text, pattern) {
  if (!pattern.length) return { index: 0, events: [] };
  const pi = prefixTable(pattern), events = [];
  let i = 0, j = 0;
  while (i < text.length) {
    const matches = text[i] === pattern[j];
    const nextMatched = matches ? j + 1 : j > 0 ? pi[j - 1] : 0;
    events.push({ textIndex: i, character: text[i], patternIndex: j,
      matches, nextMatched });
    if (matches) {
      i++; j++;
      if (j === pattern.length) return { index: i - j, events };
    } else if (j > 0) j = pi[j - 1];
    else i++;
  }
  return { index: -1, events };
}

function boyerMooreBadCharacter(text, pattern) {
  const m = pattern.length;
  if (m === 0) return 0;
  const last = new Map();
  for (let j = 0; j < m; j++) last.set(pattern[j], j);
  let start = 0;
  while (start <= text.length - m) {
    let j = m - 1;
    while (j >= 0 && pattern[j] === text[start + j]) j--;
    if (j < 0) return start;
    start += Math.max(1, j - (last.get(text[start + j]) ?? -1));
  }
  return -1;
}

function rabinKarpSearch(text, pattern) {
  const n = text.length, m = pattern.length;
  if (m === 0) return 0;
  if (m > n) return -1;
  const base = 257, modulus = 1_000_003;
  const mod = x => ((x % modulus) + modulus) % modulus;
  let factor = 1, target = 0, window = 0;
  for (let j = 0; j < m; j++) {
    target = mod(target * base + pattern.charCodeAt(j));
    window = mod(window * base + text.charCodeAt(j));
    if (j < m - 1) factor = mod(factor * base);
  }
  for (let i = 0; i <= n - m; i++) {
    if (window === target && text.startsWith(pattern, i)) return i;
    if (i < n - m) {
      window = mod((window - text.charCodeAt(i) * factor) * base
        + text.charCodeAt(i + m));
    }
  }
  return -1;
}

function buildDictionary(patterns) {
  if (patterns.some(pattern => pattern.length === 0)) {
    throw new RangeError('Use nonempty dictionary patterns.');
  }
  const nodes = [{ next: new Map(), fail: 0, output: [] }];
  for (const pattern of new Set(patterns)) {
    let state = 0;
    for (let i = 0; i < pattern.length; i++) {
      const symbol = pattern[i];
      if (!nodes[state].next.has(symbol)) {
        nodes[state].next.set(symbol, nodes.length);
        nodes.push({ next: new Map(), fail: 0, output: [] });
      }
      state = nodes[state].next.get(symbol);
    }
    nodes[state].output.push(pattern);
  }
  const queue = [...nodes[0].next.values()];
  for (let head = 0; head < queue.length; head++) {
    const state = queue[head];
    for (const [symbol, child] of nodes[state].next) {
      queue.push(child);
      let fallback = nodes[state].fail;
      while (fallback && !nodes[fallback].next.has(symbol)) fallback = nodes[fallback].fail;
      nodes[child].fail = nodes[fallback].next.get(symbol) ?? 0;
      nodes[child].output.push(...nodes[nodes[child].fail].output);
    }
  }
  return nodes;
}
function dictionarySearch(text, nodes) {
  const matches = [];
  let state = 0;
  for (let i = 0; i < text.length; i++) {
    const symbol = text[i];
    while (state && !nodes[state].next.has(symbol)) state = nodes[state].fail;
    state = nodes[state].next.get(symbol) ?? 0;
    for (const pattern of nodes[state].output) {
      matches.push({ pattern, start: i - pattern.length + 1, end: i });
    }
  }
  return matches;
}

function suffixArray(text) {
  // Straightforward teaching construction, not a linear-time suffix-array builder.
  return Array.from({ length: text.length }, (_, index) => ({ index, suffix: text.slice(index) }))
    .sort((a, b) => a.suffix < b.suffix ? -1 : a.suffix > b.suffix ? 1 : 0)
    .map(item => item.index);
}
function suffixMatches(text, indices, pattern) {
  function compare(start) {
    for (let j = 0; j < pattern.length; j++) {
      if (start + j === text.length) return -1;
      if (text[start + j] < pattern[j]) return -1;
      if (text[start + j] > pattern[j]) return 1;
    }
    return 0;
  }
  function bound(upper) {
    let low = 0, high = indices.length;
    while (low < high) {
      const mid = low + Math.floor((high - low) / 2);
      const order = compare(indices[mid]);
      if (order < 0 || (upper && order === 0)) low = mid + 1;
      else high = mid;
    }
    return low;
  }
  return indices.slice(bound(false), bound(true));
}

function gcd(a, b) {
  if (typeof a !== 'bigint' || typeof b !== 'bigint') throw new TypeError('Use BigInt inputs.');
  if (a < 0n || b < 0n) throw new RangeError('Use nonnegative inputs.');
  while (b !== 0n) [a, b] = [b, a % b];
  return a;
}
function modPow(base, exponent, modulus) {
  if ([base, exponent, modulus].some(value => typeof value !== 'bigint')) {
    throw new TypeError('Use BigInt inputs.');
  }
  if (exponent < 0n || modulus <= 0n) throw new RangeError('Require exponent >= 0 and modulus > 0.');
  base = ((base % modulus) + modulus) % modulus;
  let result = 1n % modulus;
  while (exponent > 0n) {
    if (exponent % 2n === 1n) result = result * base % modulus;
    base = base * base % modulus;
    exponent /= 2n;
  }
  return result;
}
const searches = [naiveSearch, kmpSearch, boyerMooreBadCharacter, rabinKarpSearch];
let seed = 16;
const random = () => ((seed = (Math.imul(seed, 1664525) + 1013904223) >>> 0) / 2 ** 32);
const word = length => Array.from({ length }, () => 'ABC'[Math.floor(random() * 3)]).join('');
const cases = [['', ''], ['', 'A'], ['A', ''], ['A', 'AA'], ['AAAAA', 'AAA'],
  ['ABABABC', 'ABABC'], ['BANANA', 'ANA'], ['x😀y😀', '😀'], ['abc', 'Z']];
for (let trial = 0; trial < 150; trial++) cases.push([word(Math.floor(random() * 30)), word(Math.floor(random() * 9))]);
for (const [text, pattern] of cases) {
  for (const search of searches) assert.equal(search(text, pattern), text.indexOf(pattern), search.name);
  assert.equal(kmpTrace(text, pattern).index, text.indexOf(pattern));
  const pi = prefixTable(pattern);
  for (let i = 0; i < pattern.length; i++) {
    let expected = 0;
    for (let length = 1; length <= i; length++) {
      if (pattern.slice(0, length) === pattern.slice(i + 1 - length, i + 1)) expected = length;
    }
    assert.equal(pi[i], expected);
  }
  if (pattern.length) {
    const expected = [];
    for (let i = 0; i <= text.length - pattern.length; i++) if (text.startsWith(pattern, i)) expected.push(i);
    assert.deepEqual(suffixMatches(text, suffixArray(text), pattern).sort((a, b) => a - b), expected);
  }
}
for (const text of ['', 'AAAA', 'USHERS', 'ABABABA', '😀A😀']) {
  const patterns = ['A', 'AA', 'ABA', 'HE', 'SHE', 'HERS', '😀'];
  const expected = [];
  for (let start = 0; start < text.length; start++) {
    for (const pattern of patterns) if (text.startsWith(pattern, start)) {
      expected.push({ pattern, start, end: start + pattern.length - 1 });
    }
  }
  const order = (a, b) => a.start - b.start || a.end - b.end || (a.pattern < b.pattern ? -1 : a.pattern > b.pattern ? 1 : 0);
  assert.deepEqual(dictionarySearch(text, buildDictionary(patterns)).sort(order), expected.sort(order));
}
// Check every roll against a freshly calculated hash, independently of search results.
const hash = text => { let value = 0; for (let i = 0; i < text.length; i++) value = (value * 257 + text.charCodeAt(i)) % 1000003; return value; };
const mod = value => ((value % 1000003) + 1000003) % 1000003;
const text = 'ABRACADABRA😀';
for (let m = 1; m <= text.length; m++) {
  const factor = Number(modPow(257n, BigInt(m - 1), 1000003n));
  let rolling = hash(text.slice(0, m));
  for (let start = 0; start <= text.length - m; start++) {
    assert.equal(rolling, hash(text.slice(start, start + m)));
    if (start < text.length - m) rolling = mod((rolling - text.charCodeAt(start) * factor) * 257 + text.charCodeAt(start + m));
  }
}
assert.equal(gcd(48n, 18n), 6n);
assert.equal(gcd(0n, 0n), 0n);
assert.equal(gcd(0n, 9n), 9n);
for (let a = 0n; a < 20n; a++) for (let b = 0n; b < 20n; b++) {
  let expected = 0n;
  for (let divisor = 1n; divisor <= (a > b ? a : b); divisor++) {
    if (a % divisor === 0n && b % divisor === 0n) expected = divisor;
  }
  assert.equal(gcd(a, b), expected);
}
for (let base = -5n; base <= 5n; base++) for (let exponent = 0n; exponent <= 12n; exponent++) {
  for (let modulus = 1n; modulus <= 9n; modulus++) {
    const expected = ((base ** exponent) % modulus + modulus) % modulus;
    assert.equal(modPow(base, exponent, modulus), expected);
  }
}
assert.throws(() => gcd(-1n, 2n), RangeError);
assert.throws(() => gcd(1, 2n), TypeError);
assert.throws(() => modPow(2n, -1n, 7n), RangeError);
assert.throws(() => modPow(2n, 3n, 0n), RangeError);
assert.throws(() => modPow(2, 3n, 7n), TypeError);
console.log('All Chapter 16 checks passed.');
})();

## Exercise Answers — JavaScript

Each solution below computes the answer, prints its results, and checks them with assertions. Every code cell includes its own definitions and can be run independently or rerun.


### 1. Count comparisons

For text `AAAAAA`, pattern `AAAB` tries starting indices **0, 1, and 2**. Each alignment compares all four pattern characters before the final mismatch: **12 comparisons** in total.

Pattern `BAAA` tries the same starts, but its first character fails immediately each time: **3 comparisons** in total. Both searches return **−1**.



In [ ]:
(() => {
const assert = require('node:assert/strict');
function naiveTrace(text, pattern) {
  let comparisons = 0;
  const alignments = [];
  for (let start = 0; start <= text.length - pattern.length; start++) {
    let j = 0, count = 0;
    while (j < pattern.length) {
      comparisons++; count++;
      if (text[start + j] !== pattern[j]) break;
      j++;
    }
    alignments.push({ start, comparisons: count });
    if (j === pattern.length) return { index: start, comparisons, alignments };
  }
  return { index: -1, comparisons, alignments };
}
function naiveSearch(text, pattern) { return naiveTrace(text, pattern).index; }
for (const [pattern, total] of [['AAAB', 12], ['BAAA', 3]]) {
  const result = naiveTrace('AAAAAA', pattern);
  console.log('Pattern:', pattern, 'index:', result.index, 'comparisons:', result.comparisons);
  console.table(result.alignments);
  assert.deepEqual(result.alignments.map(row => row.start), [0, 1, 2]);
  assert.equal(result.comparisons, total);
  assert.equal(result.index, -1);
}
assert.equal(naiveSearch('BANANA', 'ANA'), 1);
})();

### 2. Build a prefix table

For `ABABA`, the prefix table is **[0, 0, 1, 2, 3]**.

| Index | Pattern character | π entry |
| --- | --- | --- |
| 0 | A | 0 |
| 1 | B | 0 |
| 2 | A | 1 |
| 3 | B | 2 |
| 4 | A | 3 |

After four matched characters, a mismatch uses **π[3] = 2**. The matched segment `ABAB` ends in `AB`, which is also the pattern's first two characters. Retain those two matches and retry the same text character at pattern index 2.



In [ ]:
(() => {
const assert = require('node:assert/strict');
function prefixTable(pattern) {
  const pi = Array(pattern.length).fill(0);
  for (let i = 1, j = 0; i < pattern.length; i++) {
    while (j > 0 && pattern[i] !== pattern[j]) j = pi[j - 1];
    if (pattern[i] === pattern[j]) j++;
    pi[i] = j;
  }
  return pi;
}
function kmpSearch(text, pattern) {
  if (pattern.length === 0) return 0;
  const pi = prefixTable(pattern);
  for (let i = 0, j = 0; i < text.length; i++) {
    while (j > 0 && text[i] !== pattern[j]) j = pi[j - 1];
    if (text[i] === pattern[j]) j++;
    if (j === pattern.length) return i - j + 1;
  }
  return -1;
}
function kmpTrace(text, pattern) {
  if (!pattern.length) return { index: 0, events: [] };
  const pi = prefixTable(pattern), events = [];
  let i = 0, j = 0;
  while (i < text.length) {
    const matches = text[i] === pattern[j];
    const nextMatched = matches ? j + 1 : j > 0 ? pi[j - 1] : 0;
    events.push({ textIndex: i, character: text[i], patternIndex: j,
      matches, nextMatched });
    if (matches) {
      i++; j++;
      if (j === pattern.length) return { index: i - j, events };
    } else if (j > 0) j = pi[j - 1];
    else i++;
  }
  return { index: -1, events };
}
assert.deepEqual(prefixTable('ABABA'), [0, 0, 1, 2, 3]);
assert.equal(prefixTable('ABABA')[3], 2);
console.table([... 'ABABA'].map((character, index) => ({ index, character, pi: prefixTable('ABABA')[index] })));
const result = kmpTrace('ABABABC', 'ABABC');
console.table(result.events);
assert.equal(result.index, 2);
assert.equal(kmpSearch('ABABABC', 'ABABC'), 2);
assert.ok(result.events.some(row => row.textIndex === 4 && !row.matches && row.nextMatched === 2));
console.log('Fallback after four matches:', prefixTable('ABABA')[3]);
})();

### 3. Find the shift

At mismatch position **j = 3** in pattern `ABCA`, use `max(1, j − last(character))`.

| Text character | Rightmost pattern index | Shift |
| --- | --- | --- |
| X | −1 (absent) | max(1, 3 − (−1)) = **4** |
| B | 1 | max(1, 3 − 1) = **2** |



In [ ]:
(() => {
const assert = require('node:assert/strict');
function boyerMooreBadCharacter(text, pattern) {
  const m = pattern.length;
  if (m === 0) return 0;
  const last = new Map();
  for (let j = 0; j < m; j++) last.set(pattern[j], j);
  let start = 0;
  while (start <= text.length - m) {
    let j = m - 1;
    while (j >= 0 && pattern[j] === text[start + j]) j--;
    if (j < 0) return start;
    start += Math.max(1, j - (last.get(text[start + j]) ?? -1));
  }
  return -1;
}
const pattern = 'ABCA';
const last = new Map([...pattern].map((character, index) => [character, index]));
const shifts = ['X', 'B'].map(character => ({ character,
  last: last.get(character) ?? -1, shift: Math.max(1, 3 - (last.get(character) ?? -1)) }));
console.table(shifts);
assert.deepEqual(shifts.map(row => row.shift), [4, 2]);
assert.equal(boyerMooreBadCharacter('XXXXXXNEEDLE', 'NEEDLE'), 6);
console.log('NEEDLE starts at:', boyerMooreBadCharacter('XXXXXXNEEDLE', 'NEEDLE'));
})();

### 4. Spot a collision

Using A = 1, B = 2, base 10, and modulus 3:

- `AB`: 12 mod 3 = **0**.
- `BA`: 21 mod 3 = **0**.

The fingerprints match, but the strings do not. Character verification rejects window `BA` for pattern `AB` because the first character differs. A hash match only identifies a candidate.



In [ ]:
(() => {
const assert = require('node:assert/strict');
function rabinKarpSearch(text, pattern) {
  const n = text.length, m = pattern.length;
  if (m === 0) return 0;
  if (m > n) return -1;
  const base = 257, modulus = 1_000_003;
  const mod = x => ((x % modulus) + modulus) % modulus;
  let factor = 1, target = 0, window = 0;
  for (let j = 0; j < m; j++) {
    target = mod(target * base + pattern.charCodeAt(j));
    window = mod(window * base + text.charCodeAt(j));
    if (j < m - 1) factor = mod(factor * base);
  }
  for (let i = 0; i <= n - m; i++) {
    if (window === target && text.startsWith(pattern, i)) return i;
    if (i < n - m) {
      window = mod((window - text.charCodeAt(i) * factor) * base
        + text.charCodeAt(i + m));
    }
  }
  return -1;
}
function toyHash(word, modulus) {
  let hash = 0;
  for (const character of word) hash = (hash * 10 + character.charCodeAt(0) - 64) % modulus;
  return hash;
}
assert.equal(toyHash('AB', 3), 0);
assert.equal(toyHash('BA', 3), 0);
assert.equal('BA'.startsWith('AB'), false);
console.log('Collision:', { AB: toyHash('AB', 3), BA: toyHash('BA', 3), verified: 'BA'.startsWith('AB') });
const mod7 = value => ((value % 7) + 7) % 7;
const factor = 10 % 7;
const rolled = mod7((toyHash('AB', 7) - 1 * factor) * 10 + 3);
assert.equal(rolled, toyHash('BC', 7));
assert.equal(rolled, 2);
assert.equal(rabinKarpSearch('BANANA', 'ANA'), 1);
console.log('Rolling AB to BC:', { old: toyHash('AB', 7), factor, rolled });
console.log('Character verification rejects BA for pattern AB:', !'BA'.startsWith('AB'));
})();

### 5. Trace dictionary matching

For dictionary `HE`, `SHE`, `HERS` in text `USHERS`, using zero-based, inclusive endpoints:

| Pattern | Start index | End index |
| --- | --- | --- |
| SHE | 1 | 3 |
| HE | 2 | 3 |
| HERS | 2 | 5 |

**SHE and HE share end index 3.** Aho–Corasick reports both when it reaches the SHE state, including the HE output inherited through its failure link.



In [ ]:
(() => {
const assert = require('node:assert/strict');
function buildDictionary(patterns) {
  if (patterns.some(pattern => pattern.length === 0)) {
    throw new RangeError('Use nonempty dictionary patterns.');
  }
  const nodes = [{ next: new Map(), fail: 0, output: [] }];
  for (const pattern of new Set(patterns)) {
    let state = 0;
    for (let i = 0; i < pattern.length; i++) {
      const symbol = pattern[i];
      if (!nodes[state].next.has(symbol)) {
        nodes[state].next.set(symbol, nodes.length);
        nodes.push({ next: new Map(), fail: 0, output: [] });
      }
      state = nodes[state].next.get(symbol);
    }
    nodes[state].output.push(pattern);
  }
  const queue = [...nodes[0].next.values()];
  for (let head = 0; head < queue.length; head++) {
    const state = queue[head];
    for (const [symbol, child] of nodes[state].next) {
      queue.push(child);
      let fallback = nodes[state].fail;
      while (fallback && !nodes[fallback].next.has(symbol)) fallback = nodes[fallback].fail;
      nodes[child].fail = nodes[fallback].next.get(symbol) ?? 0;
      nodes[child].output.push(...nodes[nodes[child].fail].output);
    }
  }
  return nodes;
}
function dictionarySearch(text, nodes) {
  const matches = [];
  let state = 0;
  for (let i = 0; i < text.length; i++) {
    const symbol = text[i];
    while (state && !nodes[state].next.has(symbol)) state = nodes[state].fail;
    state = nodes[state].next.get(symbol) ?? 0;
    for (const pattern of nodes[state].output) {
      matches.push({ pattern, start: i - pattern.length + 1, end: i });
    }
  }
  return matches;
}
const nodes = buildDictionary(['HE', 'SHE', 'HERS']);
const matches = dictionarySearch('USHERS', nodes);
console.table(matches);
assert.deepEqual(matches, [
  { pattern: 'SHE', start: 1, end: 3 },
  { pattern: 'HE', start: 2, end: 3 },
  { pattern: 'HERS', start: 2, end: 5 }
]);
assert.equal(matches[0].end, matches[1].end);
assert.equal(dictionarySearch('ZZZ', nodes).length, 0);
assert.deepEqual(dictionarySearch('AAA', buildDictionary(['A', 'AA'])).map(x => x.pattern),
  ['A', 'AA', 'A', 'AA', 'A']);
assert.throws(() => buildDictionary(['']), RangeError);
console.log('Matches ending at index 3:', matches.filter(match => match.end === 3));
})();

### 6. Search a suffix array

The suffix array of `BANANA` is **[5, 3, 1, 0, 4, 2]**.

| Suffix-array position | Starting index | Suffix |
| --- | --- | --- |
| 0 | 5 | A |
| 1 | 3 | ANA |
| 2 | 1 | ANANA |
| 3 | 0 | BANANA |
| 4 | 4 | NA |
| 5 | 2 | NANA |

The suffixes beginning with `ANA` start at **[3, 1] in suffix-array order**. In text order those starting indices would be [1, 3].



In [ ]:
(() => {
const assert = require('node:assert/strict');
function suffixArray(text) {
  // Straightforward teaching construction, not a linear-time suffix-array builder.
  return Array.from({ length: text.length }, (_, index) => ({ index, suffix: text.slice(index) }))
    .sort((a, b) => a.suffix < b.suffix ? -1 : a.suffix > b.suffix ? 1 : 0)
    .map(item => item.index);
}
function suffixMatches(text, indices, pattern) {
  function compare(start) {
    for (let j = 0; j < pattern.length; j++) {
      if (start + j === text.length) return -1;
      if (text[start + j] < pattern[j]) return -1;
      if (text[start + j] > pattern[j]) return 1;
    }
    return 0;
  }
  function bound(upper) {
    let low = 0, high = indices.length;
    while (low < high) {
      const mid = low + Math.floor((high - low) / 2);
      const order = compare(indices[mid]);
      if (order < 0 || (upper && order === 0)) low = mid + 1;
      else high = mid;
    }
    return low;
  }
  return indices.slice(bound(false), bound(true));
}
const text = 'BANANA';
const indices = suffixArray(text);
console.table(indices.map(index => ({ index, suffix: text.slice(index) })));
assert.deepEqual(indices, [5, 3, 1, 0, 4, 2]);
const matches = suffixMatches(text, indices, 'ANA');
assert.deepEqual(matches, [3, 1]);
assert.deepEqual(suffixMatches(text, indices, 'APPLE'), []);
console.log('ANA starts, in suffix-array order:', matches);
})();

### 7. Trace remainder and squaring steps

Euclid's algorithm for `gcd(84n, 30n)`:

| a | b | a mod b |
| --- | --- | --- |
| 84 | 30 | 24 |
| 30 | 24 | 6 |
| 24 | 6 | 0 |

The remainders are **24, 6, 0**, so the GCD is **6**.

For `modPow(3n, 13n, 7n)`, record each state **before** its iteration:

| Exponent | Base residue | Result |
| --- | --- | --- |
| 13 | 3 | 1 |
| 6 | 2 | 3 |
| 3 | 4 | 3 |
| 1 | 2 | 5 |

In the final iteration, the exponent is odd, so the result becomes (5 × 2) mod 7 = **3**. Halving the exponent then gives zero and the loop ends.


In [ ]:
(() => {
const assert = require('node:assert/strict');
function gcd(a, b) {
  if (typeof a !== 'bigint' || typeof b !== 'bigint') throw new TypeError('Use BigInt inputs.');
  if (a < 0n || b < 0n) throw new RangeError('Use nonnegative inputs.');
  while (b !== 0n) [a, b] = [b, a % b];
  return a;
}
function modPow(base, exponent, modulus) {
  if ([base, exponent, modulus].some(value => typeof value !== 'bigint')) {
    throw new TypeError('Use BigInt inputs.');
  }
  if (exponent < 0n || modulus <= 0n) throw new RangeError('Require exponent >= 0 and modulus > 0.');
  base = ((base % modulus) + modulus) % modulus;
  let result = 1n % modulus;
  while (exponent > 0n) {
    if (exponent % 2n === 1n) result = result * base % modulus;
    base = base * base % modulus;
    exponent /= 2n;
  }
  return result;
}
let a = 84n, b = 30n;
const remainders = [];
while (b !== 0n) {
  const remainder = a % b;
  remainders.push({ a: String(a), b: String(b), remainder: String(remainder) });
  [a, b] = [b, remainder];
}
console.table(remainders);
assert.deepEqual(remainders.map(row => row.remainder), ['24', '6', '0']);
assert.equal(gcd(84n, 30n), 6n);
let base = 3n, exponent = 13n, result = 1n;
const states = [];
while (exponent > 0n) {
  states.push([exponent, base, result]);
  if (exponent % 2n === 1n) result = result * base % 7n;
  base = base * base % 7n;
  exponent /= 2n;
}
console.table(states.map(([exponent, base, result]) => ({
  exponent: String(exponent), base: String(base), result: String(result) })));
assert.deepEqual(states, [[13n, 3n, 1n], [6n, 2n, 3n], [3n, 4n, 3n], [1n, 2n, 5n]]);
assert.equal(result, 3n);
assert.equal(modPow(3n, 13n, 7n), 3n);
console.log('GCD:', String(a), 'Final modular power:', String(result));
})();

## Explain without looking

Why can the naïve search do different amounts of work for equal-length patterns? What does KMP retain after a mismatch? Why is a hash match insufficient? Which preprocessing choice suits many patterns, and which suits many queries on fixed text? Why do logarithmically many BigInt operations not necessarily imply logarithmic running time in bits?

Try changing a text or pattern and predict its first match before running the cell. If you change a worked example, also update its expected assertion. Every cell keeps definitions local so repeated runs do not cause declaration conflicts.

Adapted from the supplied Chapter 16 String and Math Algorithms text. The exercises and traces use its conventions and examples.